In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.14 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# CeO₂ on PEARL

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit the CeO₂ standard to time-of-flight neutron powder data from PEARL at ISIS: the calibration
and the profile, with a polynomial background.

It builds the catalogue case `pd-neut-tof_ceo2-pearl` step by step: the same project the command
line fits and the app's Examples open.

## Import the library

In [2]:
import easydiffraction as edi

## Define the project

A new project is empty: it holds no structure and no experiment until you add them.

In [3]:
project = edi.Project(name='pd-neut-tof_ceo2-pearl')

## Define the structure

Load the structure from EasyDiffraction's collection. Its CIF gives the space group, the cell and
the atom sites.

In [4]:
path = edi.download_data('pd-neut-tof_ceo2-pearl-ceo2', destination='data')
project.structures.add_from_cif_path(path)
structure = project.structures['ceo2']

Data 'pd-neut-tof_ceo2-pearl-ceo2' is at data/pd-neut-tof_ceo2-pearl-ceo2.edi


In [5]:
project.display.structure('ceo2')

View(html=<202,046 characters>, height=660)

## Define the experiment

Create the experiment from the measured data in EasyDiffraction's collection, then describe the
instrument, the peak profile, the background and the structure it links.

In [6]:
data_path = edi.download_data('pd-neut-tof_ceo2-pearl-pearl-data', destination='data')
project.experiments.add_from_data_path(
    name='pearl',
    data_path=data_path,
    sample_form='powder',
    beam_mode='time-of-flight',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment = project.experiments['pearl']
experiment.neutron_scattering_length = 'sears1992'
experiment.peak.type = 'tof-jorgensen-von-dreele'
experiment.absorption.type = 'cylinder'
experiment.background.type = 'polynomial'
experiment.instrument.name = 'PEARL'
experiment.instrument.facility = 'ISIS'
experiment.peak.cutoff_fwhm = 8.0
experiment.peak.rise_alpha_1 = 0.3
experiment.peak.decay_beta_0 = 0.03
experiment.peak.decay_beta_1 = 0.03
experiment.peak.broad_gauss_sigma_1 = 80
experiment.peak.broad_gauss_sigma_2 = 4
experiment.peak.broad_lorentz_gamma_1 = 5
experiment.instrument.setup_twotheta_bank = 90.0
experiment.instrument.calib_d_to_tof_linear = 4680
experiment.instrument.calib_d_to_tof_quadratic = 1.7621
experiment.background_origin = 7000.0
experiment.excluded_regions.create(start=1000.1, end=2000.0)
experiment.excluded_regions.create(start=18700.0, end=20000.0)
experiment.linked_structures.create(structure_id='ceo2', scale=2000)
terms = []
for coef in (2200.0, 0, 0, 0, 0, 0):
    term = edi.PolynomialTerm()
    term.order = len(terms)
    term.coef = coef
    terms.append(term)
experiment.background_terms = terms

Data 'pd-neut-tof_ceo2-pearl-pearl-data' is at data/pd-neut-tof_ceo2-pearl-pearl.xye


In [7]:
project.display.pattern('pearl')

View(html=<384,545 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scale and time-of-flight calibration

CeO₂ is the calibration standard, so its cell stays fixed and the DIFC terms put the peaks in
place.

In [8]:
experiment.linked_structures['ceo2'].scale.free = True
experiment.instrument.calib_d_to_tof_offset.free = True
experiment.instrument.calib_d_to_tof_linear.free = True
experiment.instrument.calib_d_to_tof_quadratic.free = True

In [9]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,pearl,instrument,,calib_d_to_tof_offset,0,0,-inf,inf,µs
2,pearl,instrument,,calib_d_to_tof_linear,4.68×10³,0,-inf,inf,µs/Å
3,pearl,instrument,,calib_d_to_tof_quadratic,1.7621,0,-inf,inf,µs/Å²
4,pearl,linked_structure,ceo2,scale,2.00×10³,0,-inf,inf,


In [10]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 4 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       14.21        27.74
      1      0.050     13.03        23.34   15.8% ↓
      2      0.062     13.03        23.34   
  done · converged · 2 iters · reduced χ² 23.34 · Rwp 13.03 % · 0.07 s


View(html=<85,716 characters>, height=376)

In [11]:
project.display.pattern('pearl')

View(html=<385,635 characters>, height=480)

### Background

The six polynomial terms follow the level between the peaks.

In [12]:
for term in experiment.background_terms:
    term.coef.free = True

In [13]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 10 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.03        23.41
      1      0.057      6.73         6.24   73.4% ↓
  done · converged · 1 iters · reduced χ² 6.24 · Rwp 6.73 % · 0.07 s


View(html=<92,522 characters>, height=376)

In [14]:
project.display.pattern('pearl')

View(html=<407,317 characters>, height=480)

### Gaussian widths

The Gaussian terms that grow with d spacing.

In [15]:
experiment.peak.broad_gauss_sigma_1.free = True
experiment.peak.broad_gauss_sigma_2.free = True

In [16]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 12 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.73         6.24
      1      0.059      6.27         5.43   13.0% ↓
  done · converged · 1 iters · reduced χ² 5.43 · Rwp 6.27 % · 0.07 s


View(html=<96,807 characters>, height=376)

In [17]:
project.display.pattern('pearl')

View(html=<407,302 characters>, height=480)

### Lorentzian width

The Lorentzian term that grows with d spacing.

In [18]:
experiment.peak.broad_lorentz_gamma_1.free = True

In [19]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 13 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.27         5.43
      1      0.059      6.19         5.30   2.5% ↓
  done · converged · 1 iters · reduced χ² 5.30 · Rwp 6.19 % · 0.07 s


View(html=<98,208 characters>, height=376)

In [20]:
project.display.pattern('pearl')

View(html=<407,190 characters>, height=480)

### Rise and decay

The back-to-back exponentials that shape each time-of-flight peak.

In [21]:
experiment.peak.rise_alpha_1.free = True
experiment.peak.decay_beta_0.free = True
experiment.peak.decay_beta_1.free = True

In [22]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 16 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.19         5.30
      1      0.055      5.67         4.44   16.3% ↓
      2      0.069      5.67         4.44   
  done · converged · 2 iters · reduced χ² 4.44 · Rwp 5.67 % · 0.08 s


View(html=<101,869 characters>, height=376)

In [23]:
project.display.pattern('pearl')

View(html=<407,193 characters>, height=480)

### Displacements

In [24]:
structure.atom_sites['Ce'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True

In [25]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 18 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.67         4.44
      1      0.062      5.29         3.88   12.7% ↓
  done · converged · 1 iters · reduced χ² 3.88 · Rwp 5.29 % · 0.07 s


View(html=<104,581 characters>, height=376)

In [26]:
project.display.pattern('pearl')

View(html=<407,190 characters>, height=480)

### Absorption

Absorption weakens the intensities with time of flight, as the displacements do, so it comes
after them, on its own.

In [27]:
experiment.absorption.abscor1.free = True

In [28]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 19 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.29         3.88
      1      0.050      5.29         3.88   
  done · converged · 1 iters · reduced χ² 3.88 · Rwp 5.29 % · 0.06 s


View(html=<104,983 characters>, height=376)

In [29]:
project.display.pattern('pearl')

View(html=<407,180 characters>, height=480)

In [30]:
project.display.fit.correlations()

,1,2,3,4,5
1 pearl.linked_structure.ceo2.scale,,,,,
2 pearl.background.3.coef,-0.04,,,,
3 pearl.background.5.coef,0.19,-0.92,,,
4 pearl.background.6.coef,-0.14,0.89,-0.95,,
5 pearl.absorption.abscor1,0.96,-0.05,0.18,-0.14,


In [31]:
project.display.structure('ceo2')

View(html=<206,052 characters>, height=660)

## Report

The HTML report draws the measured, calculated and difference patterns with the reflection
positions, and lists the structures, the experiments and the fit. Open it in a browser.

In [32]:
project.report.save_html('pd-neut-tof_ceo2-pearl.html')

Saving report to 'pd-neut-tof_ceo2-pearl.html'


## Save the project

In [33]:
project.save_as('pd-neut-tof_ceo2-pearl')

Saving project to 'pd-neut-tof_ceo2-pearl'
